**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/du-2026"], check=True)
    os.chdir("/content/encore/entries/du-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Du, Lai, Hu & Ye (2026) on LINK: the 20 confirmation sessions

Paper doi:10.64898/2026.08.04.742876; dataset DANDI:001201 version 0.251023.2336 (the version the
paper cites; see `02_link_development.ipynb` for the full header and claims table).

The confirmation sessions are re-selected here with the paper's rule (Methods 4.2; port of
`build_link_confirmation_manifest.py`) from the archive's asset metadata, and compared with the
paper's frozen manifest (`outputs/primary_analysis/link_confirmation_manifest.json` in the release at
commit b042fbe, read from GitHub; an input).

| Claim | Status |
|---|---|
| confirm-geometry-gain | tested |
| confirm-history-contraction | tested |
| confirm-replacement | tested |
| confirm-table1-values | tested |

In [ ]:
import collections, resource, time, zlib
import numpy as np
import pandas as pd
import du2026 as d

PAPER = "du-2026"
def claim_seed(claim_id):
    return zlib.crc32(f"{PAPER}/{claim_id}".encode())
T0 = time.time()
EXTRA = (1000, 2000, 3000, 4000)

assets = d.list_assets(d.LINK_ID, d.LINK_VERSION)
by_date = collections.defaultdict(list)
for a in assets:
    by_date[d.session_date(a["path"])].append(a)
paired_dates = sorted(k for k, v in by_date.items() if len(v) > 1)
dev_assets = [a for k in paired_dates for a in sorted(by_date[k], key=lambda a: a["path"])]
print(f"{len(assets)} assets in DANDI:{d.LINK_ID} {d.LINK_VERSION}; {len(paired_dates)} dates with two sessions, {len(dev_assets)} development sessions")

def table(results, name, field="gain"):
    return pd.DataFrame([{"date": r["date"], "task": r["task"], field: r[name][field]} for r in results])

from concurrent.futures import ThreadPoolExecutor
import json, urllib.request
with ThreadPoolExecutor(12) as pool:
    metadata = dict(zip([a["asset_id"] for a in assets],
                        pool.map(lambda a: d.asset_metadata(d.LINK_ID, d.LINK_VERSION, a["asset_id"]), assets)))
dev_ids = {a["asset_id"] for a in dev_assets}
records = [dict(asset_id=a["asset_id"], path=a["path"], date=d.session_date(a["path"]), task=d.link_task_from_metadata(metadata[a["asset_id"]]))
           for a in assets]
print("tasks over all 312 assets:", collections.Counter(r["task"] for r in records))
selected = []
for task in ("CO", "RD"):
    eligible = sorted((r for r in records if r["task"] == task and r["asset_id"] not in dev_ids), key=lambda r: (r["date"], r["asset_id"]))
    ranks = d.interior_ranks(len(eligible))
    print(f"{task}: {len(eligible)} eligible assets, ranks {ranks}")
    selected += [eligible[k] for k in ranks]
url = f"https://raw.githubusercontent.com/gwkuqgfkqe/neural-decoder-context-audit/{d.PAPER_COMMIT}/outputs/primary_analysis/link_confirmation_manifest.json"
frozen = json.load(urllib.request.urlopen(url, timeout=60))
frozen_ids = [r["asset_id"] for r in frozen.get("selected", frozen.get("sessions", []))]
print(f"frozen manifest: {len(frozen_ids)} assets; re-selected here: {len(selected)}; identical sets: {set(frozen_ids) == {r['asset_id'] for r in selected}}")
print("in the frozen manifest only:", set(frozen_ids) - {r['asset_id'] for r in selected}, "; re-selected only:", {r['asset_id'] for r in selected} - set(frozen_ids))
print(pd.DataFrame(selected)[["date", "task", "path"]].to_string())

In [ ]:
conf_assets = [a for a in assets if a["asset_id"] in set(frozen_ids)]
sessions = [d.load_link_session(a["asset_id"], a["path"], signals=("SpikingBandPower",)) for a in conf_assets]
chk = []
for s in sessions:
    start_idx = np.array([int(np.argmin(np.abs(s.timestamps - t))) for t in s.trials.start_time])
    usable = int(np.sum(start_idx + 8 + 24 < np.minimum(start_idx + s.trials.trial_count.to_numpy(), len(s.timestamps))))
    chk.append(dict(session=s.identifier, task=s.task, trials=len(s.trials), usable=usable, bins=len(s.timestamps),
                    dt_ms=1000 * float(np.median(np.diff(s.timestamps))),
                    sbp=f"{s.signals['SpikingBandPower'].min():.3g}..{s.signals['SpikingBandPower'].max():.3g}",
                    pos=f"{s.position.min():.3g}..{s.position.max():.3g}", vel=f"{s.velocity.min():.3g}..{s.velocity.max():.3g}"))
chk = pd.DataFrame(chk)
print(chk.to_string())
print("tasks:", collections.Counter(chk.task), "; usable trials", chk.usable.sum(), "(paper 7,500); sessions with 375 usable:", int((chk.usable == 375).sum()))
print("task in the NWB identifier agrees with the asset metadata:",
      all(s.task == d.link_task_from_metadata(metadata[s.asset_id]) for s in sessions))
import hashlib
h = [hashlib.sha1(s.signals["SpikingBandPower"].tobytes()).hexdigest() for s in sessions]
print("distinct SBP arrays:", len(set(h)), "of", len(h))
del sessions
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## The fits

Fully nested estimator, SBP, geometry by phase and plus history, outer split seed 20260726
(`run_link_confirmation.py`), and the same under split and refit seeds plus 1000 to 4000 (protocol,
seed sensitivity).

In [ ]:
SEED = 20260726
jobs = []
for k in (0,) + EXTRA:
    for context in ("geometry_tensor", "causal_history"):
        jobs.append((f"{context}_{SEED + k}", dict(signal="SpikingBandPower", context_mode=context, seed=SEED + k), "nested", {"seed": SEED + k}, False))
t = time.time()
results = d.run_parallel(d.link_session_jobs, [(a["asset_id"], a["path"], jobs, ("SpikingBandPower",)) for a in conf_assets])
print(f"{len(jobs)} fits per session on {len(results)} sessions in {(time.time() - t) / 60:.1f} min")
G, H = f"geometry_tensor_{SEED}", f"causal_history_{SEED}"
per = table(results, G).rename(columns={"gain": "geometry"})
per["history"] = table(results, H).gain
per["replacement_geometry"] = table(results, G, "replacement_gain").replacement_gain
per["replacement_history"] = table(results, H, "replacement_gain").replacement_gain
per["mse_red_geometry"] = table(results, G, "mse_reduction_percent").mse_reduction_percent
per["mse_red_history"] = table(results, H, "mse_reduction_percent").mse_reduction_percent
print(per.round(5).to_string())
def seed_line(field, ctx):
    out = []
    for k in (0,) + EXTRA:
        t_ = table(results, f"{ctx}_{SEED + k}", field)
        out.append(f"{t_[field].mean():.5f} ({int((t_[field] > 0).sum())} positive)")
    return "; ".join(out)

## confirm-geometry-gain

> In the 20 untouched LINK confirmation sessions, SBP geometry-by-phase gain was positive in all 20
> and averaged 0.02158 (CO 0.02104, RD 0.02213).

In [ ]:
m = per.groupby("task").geometry.mean()
print(f"positive {int((per.geometry > 0).sum())}/20 (expected 20, tolerance 2); mean {per.geometry.mean():.5f} (0.02158 +- 0.002158); "
      f"CO {m['CO']:.5f} (0.02104 +- 0.002104); RD {m['RD']:.5f} (0.02213 +- 0.002213); range {per.geometry.min():.5f} to {per.geometry.max():.5f}")
print("paper's gate (at least 16 of 20 positive, positive CO and RD means):", bool((per.geometry > 0).sum() >= 16 and m["CO"] > 0 and m["RD"] > 0))
print("combined mean under seeds 20260726 + 0/1000/2000/3000/4000:", seed_line("gain", "geometry_tensor"))

**Result: reproduced.** The paper's rule re-run on the archive's asset metadata selects the same 20 sessions as its frozen manifest. Under geometry by phase all 20 gains are positive and the mean is 0.02158 (CO 0.02104, RD 0.02213), the paper's values. The values depend on the split: under the four other split seeds the combined mean is 0.01478 to 0.02070, with 19 or 20 sessions positive, so three of those seeds put the mean outside its 10% tolerance.

## confirm-history-contraction

> In the confirmation sessions, adding measured-output history reduced the combined mean gain to
> 0.00755 (CO 0.00670, RD 0.00840), with 19 of 20 sessions positive.

In [ ]:
m = per.groupby("task").history.mean()
print(f"mean {per.history.mean():.5f} (0.00755 +- 0.000755); CO {m['CO']:.5f} (0.0067 +- 0.00067); RD {m['RD']:.5f} (0.0084 +- 0.00084); "
      f"positive {int((per.history > 0).sum())}/20 (19 +- 1.9); below the geometry mean {per.geometry.mean():.5f}: {per.history.mean() < per.geometry.mean()}")
print(f"range {per.history.min():.5f} to {per.history.max():.5f}")
print("combined mean under the five seeds:", seed_line("gain", "causal_history"))

**Result: reproduced.** With measured-output history the combined mean is 0.00755 (CO 0.00670, RD 0.00840) with 19 of 20 sessions positive, below the geometry-by-phase mean of 0.02158. Under the four other split seeds the combined mean is 0.00335 to 0.00818, three of them outside the tolerance.

## confirm-replacement

> In the confirmation sessions, nearest-replacement gain averaged -0.00771 under geometry by phase,
> and correct-pair gain exceeded replacement gain in all 20 sessions and in both task means.

In [ ]:
above = int((per.geometry > per.replacement_geometry).sum())
mt = per.groupby("task")[["geometry", "replacement_geometry"]].mean()
print(f"mean replacement gain {per.replacement_geometry.mean():.5f} (-0.00771 +- 0.000771); sessions with gain above replacement {above}/20 (20 +- 2)")
print(mt.round(5).to_string())
print("correct-pair mean above replacement in CO and RD:", bool((mt.geometry > mt.replacement_geometry).all()))
print("replacement mean under the five seeds:", seed_line("replacement_gain", "geometry_tensor"))
for k in (0,) + EXTRA:
    g_, r_ = table(results, f"geometry_tensor_{SEED + k}"), table(results, f"geometry_tensor_{SEED + k}", "replacement_gain")
    tm = pd.DataFrame({"task": g_.task, "g": g_.gain, "r": r_.replacement_gain}).groupby("task").mean()
    print(f"seed {SEED + k}: sessions with gain above replacement {int((g_.gain > r_.replacement_gain).sum())}/20; "
          f"task means above replacement in CO and RD: {bool((tm.g > tm.r).all())}")

**Result: reproduced.** The mean nearest-replacement gain is -0.00771, and the correct-pair gain exceeds the replacement gain in 20 of 20 sessions and in both task means (CO 0.02104 against -0.00292, RD 0.02213 against -0.01250). Under the four other split seeds the replacement mean is -0.00902 to -0.01963, outside the 10% tolerance in each; the correct-pair gain exceeds the replacement gain in 20, 19, 20 and 20 of 20 sessions and in both task means under each.

## confirm-table1-values

> Table 1 gives, for the confirmation sessions, a replacement gain of -0.00498 with measured-output
> history and MSE reductions of 4.76% (geometry by phase) and 2.05% (plus history).

In [ ]:
print(f"replacement gain, plus history: {per.replacement_history.mean():.5f} (-0.00498 +- 0.000498)")
print(f"MSE reduction, geometry: {per.mse_red_geometry.mean():.3f}% (4.76 +- 0.476); plus history: {per.mse_red_history.mean():.3f}% (2.05 +- 0.205)")
print("plus-history replacement under the five seeds:", seed_line("replacement_gain", "causal_history"))
print("geometry MSE reduction under the five seeds:", seed_line("mse_reduction_percent", "geometry_tensor"))
print("history MSE reduction under the five seeds:", seed_line("mse_reduction_percent", "causal_history"))
print(f"run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: reproduced.** With measured-output history the mean replacement gain is -0.00498, and the mean MSE reductions are 4.761% (geometry by phase) and 2.047% (plus history), the values of Table 1. Under the four other split seeds these become -0.00467 to -0.01462, 3.43 to 4.58% and 1.15 to 2.21%, mostly outside their tolerances.

## Summary

| Claim | Outcome |
|---|---|
| confirm-geometry-gain | reproduced (seed sensitive) |
| confirm-history-contraction | reproduced (seed sensitive) |
| confirm-replacement | reproduced (seed sensitive) |
| confirm-table1-values | reproduced (seed sensitive) |

The confirmation sessions re-selected from the archive's metadata are those of the paper's frozen
manifest; each has 375 usable trials (7,500 in all) and no SBP array repeats. Every value equals the
paper's under its split seed 20260726. Under the four other split seeds most values fall outside their
10% tolerances (the combined geometry and history means are inside only under seed plus 3000, and the
replacement mean under none), while the qualitative pattern holds in the printed seed runs: positive geometry gain, a smaller history gain,
and replacement below correct pairing in both task means and in 19 or 20 of the 20 sessions.